In [1]:
from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

# Launch DPS for `zonal_stats_landscapes.py` & `calculate_biomass_change.py`

In [2]:
pip install contextily

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import geopandas as gpd
import pandas as pd
import glob
import datetime
import time

import sys
sys.path.append('/projects/code/icesat2_boreal/lib')
import ExtractUtils, zonal_stats_landscapes
import mosaiclib
import contextily as ctx
import importlib

NASA MAAP


In [4]:
mosaiclib.AGB_TINDEX_FN_DICT

{'2019_v1.9': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v5/AGB_H30_2019/2019_fullboreal_2019lidar/AGB_tindex_master.csv',
 '2020_v1.9': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v5/AGB_H30_2020/atl08_v6_fullboreal_min5000_90p_local/AGB_tindex_master.csv',
 '2021_v1.9': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v5/AGB_H30_2021/2021_fullboreal_2021lidar/AGB_tindex_master.csv',
 '2022_v1.9': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v5/AGB_H30_2022/2022_fullboreal_2022lidar/AGB_tindex_master.csv',
 '2023_v1.9': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v5/AGB_H30_2023/2023_full_2023lidar/AGB_tindex_master.csv',
 '2020_v2.0': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/boreal_agb_2024_v6/AGB_H30_2020/Version2_SD/AGB_tindex_master.csv',
 '2020_v2.1': 's3://maap

## Steps to commit, build DPS registration yaml, register DPS algorithm from yaml
### Commit with Tag for running
1) Tag the version of the repo that works to run your alg. Use a *tag* of `build_stack_v2024_1` or whatever is appropriate (eg, for AGB runs maybe do `boreal_agb_2023_v1` for consistency?)
  - to maintain sanity, use this GitHub *tag* also as the `algorithm_version` that you need to supply to your algorithm config yaml  
2) think now about how you want your output organized:  
  - remember, the output will be like: `dps_output/<algorithm name>/<algorithm_version>/<IDENTIFIER>`  
  - note: identifier for biomass runs should be `AGB_2020` , for height `HT_2020`  
  - if different types of `AGB_2020` runs (eg like is you are testing different sets of parameters associated with model dev/application you can keep the results separate from one run to the next, delivering the output into different subdirs, by modifying this IDENTIFIER like this: `AGB_2020/run_param_set01`     
    
  
3) follow git instructions (every time!!):  
 - git add changes  
 - git commit -m 'message'  
 - git tag -f `zonal_stats_landscapes_v1`    
 - git push  
 - git push origin -f `zonal_stats_landscapes_v1`  

3) if it looks weird check git log to make sure tag is at same place as origin and dps

### Use MAAP Registration call in notebook chunk to register DPS algorithm
 - We need to register the DPS algorithm before proceeding to the chunks below...

In [39]:
maap.register_algorithm_from_yaml_file("/projects/code/icesat2_boreal/dps/registered/run_zonal_stats_landscapes_v1.yml").text

'{"code": 200, "message": {"id": "42000cf919ad6ff1e843a1daeac312aaa570ccba", "short_id": "42000cf9", "created_at": "2025-12-09T13:38:50.000+00:00", "parent_ids": ["dc1f723f05b724e0d87b0e184f630c94b339d144"], "title": "Registering algorithm: run_zonal_stats_landscapes", "message": "Registering algorithm: run_zonal_stats_landscapes", "author_name": "root", "author_email": "root@971fce4a40be", "authored_date": "2025-12-09T13:38:50.000+00:00", "committer_name": "root", "committer_email": "root@971fce4a40be", "committed_date": "2025-12-09T13:38:50.000+00:00", "trailers": {}, "extended_trailers": {}, "web_url": "https://repo.maap-project.org/root/register-job-hysds-v4/-/commit/42000cf919ad6ff1e843a1daeac312aaa570ccba", "stats": {"additions": 7, "deletions": 7, "total": 14}, "status": "pending", "project_id": 3, "last_pipeline": {"id": 16469, "iid": 2864, "project_id": 3, "sha": "42000cf919ad6ff1e843a1daeac312aaa570ccba", "ref": "hysds-v5", "status": "pending", "source": "push", "created_at":

# Build a DPS list

## Setup DPS run info

### Select hydrobasins by filtering to only those that intersect our TREND tiles

In [5]:
boreal_tiles = gpd.read_file(mosaiclib.boreal_tile_index_path)

In [6]:
HYBAS_FULL_HILAT_FN = 'https://maap-ops-workspace.s3.amazonaws.com/shared/montesano/databank/hydrobasins/hybas_lake_hilat40N_75N_lev08_v1c.gpkg'

In [7]:
hybas = gpd.read_file(HYBAS_FULL_HILAT_FN) # Read in the full file

In [11]:
hybas.shape

(89309, 16)

In [12]:
#HYBAS_FN = '/projects/my-public-bucket/databank/hydrobasins/hybas_lake_na_lev08_v1c.shp'
HYBAS_FN_STEM = os.path.basename(HYBAS_FULL_HILAT_FN).split('.')[0]

In [13]:
# These are the trendclass tiles that are ready for testing zonal stats with hydrobasins
TREND_TILES = pd.read_csv(mosaiclib.MISC_TINDEX_FN_DICT['TRENDCLASS_AGB_v3.1_multiyr_2020-2025']).tile_num.to_list()

In [14]:
import age_trend_functions

In [15]:
%%time
# Filter to those basins that intersect the set of TREND tiles
hybas_filt = age_trend_functions.filter_intersecting_hydrobasins_spatial_index(hybas, boreal_tiles[boreal_tiles.tile_num.isin(TREND_TILES)], buffer_degrees=0.0001, verbose=False)
hybas_filt.shape

CPU times: user 25min 24s, sys: 21.5 s, total: 25min 46s
Wall time: 25min 21s


(64892, 16)

In [16]:
hybas_filt.to_file('/projects/my-public-bucket/databank/hydrobasins/hybas_lake_hilat40N_75N_lev08_v1c_TREND_TILES_intersect.gpkg')

### Build ID list for DPS

In [29]:
DPS_HYBAS_ID_LIST = hybas_filt.HYBAS_ID.to_list()
len(DPS_HYBAS_ID_LIST)

64892

#### Mosaics for zonal: CACC 2020, AGB trends, AGB 2025

In [30]:
MOS_LIST = [m.replace('_mosaic.json','.json') for m in [mosaiclib.MISC_MOSAIC_JSON_FN_DICT['CACC_2020_v3.1_multiyr_nsims050'], 
                                                        mosaiclib.MISC_MOSAIC_JSON_FN_DICT['TRENDCLASS_AGB_v3.1_multiyr_2020-2025'], 
                                                        mosaiclib.AGB_MOSAIC_JSON_FN_DICT['2020_v3.1_multiyr'],
                                                        mosaiclib.AGB_MOSAIC_JSON_FN_DICT['2025_v3.1_multiyr']
                                                       ]
           ]
MOS_STRINGS = ','.join(MOS_LIST)
PREFIX_STRINGS = ','.join([os.path.basename(os.path.dirname(p)) for p in MOS_LIST[0:2]]) +','
PREFIX_STRINGS += ','.join([os.path.basename(os.path.dirname(os.path.dirname(p))) for p in MOS_LIST[2:]])

#### Note: make sure the `in_params_dict` coincides with the args in run.sh that runs `zonal_stats_landscapes.py` in a particular way

In [31]:
# cmd = f'/opt/conda/envs/pangeo/bin/python /projects/code/icesat2_boreal/lib/zonal_stats_landscapes.py -h'
# !eval $cmd

In [32]:
in_params_dict = {
    # These are for zonal_stats_landscapes.py
      'polygons': HYBAS_FULL_HILAT_FN, # feed in polygon-ids list using BATCH_SIZE
      'polygon-ids': '' ,
      'polygon-id-col': 'HYBAS_ID',
      #'output': 'results.gpkg',
      'mosaic-geojsons': MOS_STRINGS,
      'zs-col-prefix': PREFIX_STRINGS,
      'bands': '1,1,1,1',
      'statistics': 'sum,count',
      'age-classes': 'non-forest:0-0 young:1-20 maturing:21-80 mature:81-200 old-growth:201-1000',
      'age-mosaic-index': 0,
      'age-band': 9,
      'trend-mosaic-index': 1,
      'trend-band': 1,
      'additional-biomass-indices': '2,2,3,3',
      'additional-biomass-bands': '1,2,1,2',
      'additional-biomass-names': 'agb_2020_mean,agb_2020_std,agb_2025_mean,agb_2025_std',
      'processes': 30,
      'chunk-size': 50, # not sure if better to have chuunk size always at 1 - especially for runs with num polygons < num processes
    
    # These are for calculate_biomass_change.py
    'year1-prefix': 'agb_2020',
    'year2-prefix': 'agb_2025',
    'simulations': 1000,

    # This is for zonal_stats_landscapes_misc.py
    'uncertainty-pairs': 'cacc_2020:0,1,0,2'
            }
in_params_dict

{'polygons': 'https://maap-ops-workspace.s3.amazonaws.com/shared/montesano/databank/hydrobasins/hybas_lake_hilat40N_75N_lev08_v1c.gpkg',
 'polygon-ids': '',
 'polygon-id-col': 'HYBAS_ID',
 'mosaic-geojsons': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/CACC/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/CACC_tindex_master.json,s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/TRENDCLASS/compute_trends_ols_v3/AGB_v3.1_multiyr_2020-2025/TRENDCLASS_tindex_master.json,s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/v3.1.0/AGB_H30_2020/full_run_niter30_multiyear_atl08v6/AGB_tindex_master.json,s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP/v3.1.0/AGB_H30_2025/full_run_niter30_multiyear_atl08v6/AGB_tindex_master.json',
 'zs-col-prefix': 'CACC_2020_v3.1_multiyr_nsims050,AGB_v3.1_multiyr_2020-2025,AGB_H30_2020,AGB_H30_2025',
 'bands': '1,1,1,1',
 'statistics': 'sum,count',
 'age-classes': 'non-forest:0-0 young:1-20 maturing:21-80 

In [33]:
# MAAP algorithm version name
IDENTIFIER = "agb_trends_2020_2025" 
MAAP_VERSION = 'zonal_stats_landscapes_v1'
ALGO_ID      = 'run_zonal_stats_landscapes'
USER = 'montesano'
WORKER_TYPE = 'maap-dps-gedi_boreal_worker-16gb' # 'maap-dps-gedi_boreal_worker-32gb'

In [34]:
RUN_NAME = IDENTIFIER
print(f"{ALGO_ID}, {MAAP_VERSION}, {RUN_NAME}")

run_zonal_stats_landscapes, zonal_stats_landscapes_v1, agb_trends_2020_2025


## Run a DPS job across the list

In [35]:
BATCH_SIZE = 30 # this can be equal to the # of simultaneous processes you want running for a single job (on a single machine)
WAIT_MINS = 0
IDENTIFIER

'agb_trends_2020_2025'

In [36]:
TEST = True

In [55]:
if TEST:
    DPS_INPUT_LIST = DPS_HYBAS_ID_LIST[3000:]
else:
    DPS_INPUT_LIST = DPS_HYBAS_ID_LIST

len(DPS_INPUT_LIST)

61892

In [56]:
import math
print(f'{math.floor((len(DPS_INPUT_LIST)/BATCH_SIZE)+1)} DPS jobs will be run')

2064 DPS jobs will be run


In [57]:
import json

In [58]:
%%time

submit_results_df_list = []
len_input_list = len(DPS_INPUT_LIST)

print(f"# of input HYBAS_ID for DPS: {len_input_list}")

for i in range(0, len_input_list, BATCH_SIZE):
    
    INPUT_TILE_NUM = -1
    DPS_HYBAS_ID_LIST_BATCH = DPS_HYBAS_ID_LIST[i:i + BATCH_SIZE]
    
    DPS_num = i+1
    
    # Update the in_params_dict with th current INPUT_TILE_NUM
    in_params_dict['polygon-ids'] = ','.join([str(i) for i in DPS_HYBAS_ID_LIST_BATCH])
    
    submit_result = maap.submitJob(
            identifier=IDENTIFIER,
            algo_id=ALGO_ID,
            version=MAAP_VERSION,
            username=USER, # username needs to be the same as workspace owner
            queue=WORKER_TYPE,
            **in_params_dict
        )
    
    # Build a dataframe of submission details
    submit_result_df = pd.DataFrame( 
        {
                'dps_num':[DPS_num],
                'tile_num':[INPUT_TILE_NUM],
                'submit_time':[datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%s')],
                'dbs_job_hour': [datetime.datetime.now().hour],
                'algo_id': [ALGO_ID],
                'user': [USER],
                'worker_type': [WORKER_TYPE],
                'job_id': [submit_result.id],
                'submit_status': [submit_result.status],
            
        } 
    )
    
    # Append to a list of data frames of submission results
    submit_results_df_list.append(submit_result_df)
    
    if DPS_num in [1, 5, 10, 30, 50, 100, 250, 300, 500, 600 , 750, 900, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 7000, 9000, 11000, 13000, 15000, 17000, 19000, 21000, 24000, len_input_list]:
        print(f"DPS run #: {DPS_num}\t| tile num: {INPUT_TILE_NUM}\t| submit status: {submit_result.status}\t| job id: {submit_result.id}") 

    # Choose a BATCH_SIZE that plays nice with EarthData - don try to do too many simultaneous MS comp jobs...
    if DPS_num in list(range(BATCH_SIZE, len(DPS_HYBAS_ID_LIST), BATCH_SIZE)):
        print(f'...waiting {WAIT_MINS} minutes...')
        time.sleep(60 * WAIT_MINS) # sleep time in seconds
        
# Build a final submission results df and save
submit_results_df = pd.concat(submit_results_df_list)
submit_results_df['run_name'] = RUN_NAME
nowtime = pd.Timestamp.now().strftime('%Y%m%d%H%M')
print(f"Current time:\t{nowtime}")
submit_results_df.to_csv(f'/projects/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{RUN_NAME}_submission_results_{len_input_list}_{nowtime}.csv')
submit_results_df.info()

# of input HYBAS_ID for DPS: 61892
DPS run #: 1	| tile num: -1	| submit status: success	| job id: 6206d324-5728-4085-80ff-b324efe33f9c
Current time:	202601131223
<class 'pandas.core.frame.DataFrame'>
Index: 2064 entries, 0 to 0
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   dps_num        2064 non-null   int64 
 1   tile_num       2064 non-null   int64 
 2   submit_time    2064 non-null   object
 3   dbs_job_hour   2064 non-null   int64 
 4   algo_id        2064 non-null   object
 5   user           2064 non-null   object
 6   worker_type    2064 non-null   object
 7   job_id         2064 non-null   object
 8   submit_status  2064 non-null   object
 9   run_name       2064 non-null   object
dtypes: int64(3), object(7)
memory usage: 177.4+ KB
CPU times: user 24 s, sys: 4.9 s, total: 28.9 s
Wall time: 4min 36s


After almost any DPS job, you have to assess what was marked as `success` and `fail`.  

This involves:
1. building a table of job status based on job ids captured in the job_results_df from the DPS run chunk (this takes 40 mins for ~47k jobs). This tells you how many jobs failed.
2. merging the `job status table` with the `job results df`. This tells you which specific granules (or tile nums) failed.
3. building another input list of granules/tiles for a follow-up DPS run.
## Assess DPS results
Build a table of job status based on job id - how many jobs failed?

In [60]:
LIST_SUBMISSIONS = sorted(glob.glob(f'/projects/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_*_submission_results_*.csv'),key=ExtractUtils.func, reverse=True)
LIST_SUBMISSIONS[0:5]

['/projects/my-public-bucket/dps_submission_results/DPS_run_zonal_stats_landscapes_agb_trends_2020_2025_submission_results_61892_202601131223.csv',
 '/projects/my-public-bucket/dps_submission_results/DPS_run_zonal_stats_landscapes_agb_trends_2020_2025_submission_results_3000_202601131039.csv',
 '/projects/my-public-bucket/dps_submission_results/DPS_run_zonal_stats_landscapes_agb_trends_2020_2025_submission_results_90_202601071314.csv',
 '/projects/my-public-bucket/dps_submission_results/DPS_run_zonal_stats_landscapes_agb_trends_2020_2025_submission_results_3000_202512090850.csv',
 '/projects/my-public-bucket/dps_submission_results/DPS_run_zonal_stats_landscapes_agb_trends_2020_2025_submission_results_300_202512090558.csv']

In [61]:
%%time

running_list = []
fails_list = []
success_list = []
offline_list = []

#for DPS_DATETIME in [nowtime]:
for fn in LIST_SUBMISSIONS[0:1]:
    #if DPS_DATETIME in fn and not 'job_status' in fn:

    DPS_alg_id = os.path.basename(fn.split('_submission_results_')[0].replace('DPS_',''))
    thentime = fn.split('_')[-1].replace('.csv','')
    print(f'DPS alg:\t\t{DPS_alg_id}')
    #print(f'DPS run name:\t\t{RUN_NAME}')
    print(f'DPS launch time:\t{thentime}')

    # Build job status table
    df_jstatus = ExtractUtils.BUILD_TABLE_JOBSTATUS(pd.read_csv(fn))

    # Save job status table
    df_jstatus.to_csv(f'/projects/my-public-bucket/dps_submission_results/DPS_{RUN_NAME}_submission_results_job_status_{len(df_jstatus)}_{thentime}.csv')

    # Get current fails df and append to list
    df_jstatus['run_type'] = RUN_NAME

    running_list.append(df_jstatus[ (df_jstatus['status'] == 'Running') ] )
    fails_list.append(  df_jstatus[ (df_jstatus['status'] == 'Failed') ] )
    success_list.append(df_jstatus[ (df_jstatus['status'] == 'Succeeded') ] )
    offline_list.append(df_jstatus[ (df_jstatus['status'] == 'Offline') ] )
    print(f"Count offline jobs: {df_jstatus[ (df_jstatus['status'] == 'Offline') ].shape[0]}\n")
            
df_all_running = pd.concat(running_list)          
df_all_fails =   pd.concat(fails_list)
df_all_success = pd.concat(success_list)
df_all_offline = pd.concat(offline_list)

DPS alg:		run_zonal_stats_landscapes_agb_trends_2020_2025
DPS launch time:	202601131223
multiprocessing...
Count total jobs:	2064
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	2060
Count failed jobs:	2
% of failed jobs:	0.1

Count offline jobs: 1

CPU times: user 474 ms, sys: 734 ms, total: 1.21 s
Wall time: 12.8 s


In [62]:
#!aws s3 rm --recursive s3://maap-ops-workspace/montesano/dps_output/run_tile_forestage/

In [63]:
# New list = fails + still running + failed to even submit
FAILS = df_all_fails.tile_num.to_list() +\
        df_all_running.tile_num.to_list() +\
        df_all_offline.tile_num.to_list() +\
        list(set(df_jstatus[df_jstatus.submit_status == 'failed'].tile_num.to_list()))
FAILS = [int(i) for i in FAILS]
print(len(FAILS))
#print(FAILS)
DPS_INPUT_TILE_NUM_LIST=FAILS

4


In [64]:
print(DPS_INPUT_TILE_NUM_LIST)
df_all_fails

[-1, -1, -1, -1]


,Unnamed: 0,dps_num,tile_num,submit_time,dbs_job_hour,algo_id,user,worker_type,job_id,submit_status,run_name,status,run_type
1243,0,37291,-1,2026-01-13-12-21-1768335711,12,run_zonal_stats_landscapes,montesano,maap-dps-gedi_boreal_worker-16gb,58b919ab-c7cb-4b76-8077-93a5fd200663,success,agb_trends_2020_2025,Failed,agb_trends_2020_2025
1605,0,48151,-1,2026-01-13-12-22-1768335760,12,run_zonal_stats_landscapes,montesano,maap-dps-gedi_boreal_worker-16gb,86a1038c-9284-47f8-96d1-5efdbbc0407f,success,agb_trends_2020_2025,Failed,agb_trends_2020_2025
